# Testing Effect (C) — Stage 2 Data Prep, retrieval practice folded into curation

Mirrors `06b`'s pseudo-document / cost-gate / curate / retention-check structure exactly, with one change: `curate_document_threaded` is replaced by `curate_document_threaded_with_testing` (`src/pipeline/teacher.py`) — after every chunk's write-back, self-test against the same evidence-linked questions `06b` already builds into `answers_by_chunk`, corrective-retry on failure, verbatim fallback if retries are exhausted.

**Why this counts as "testing effect", psychologically**: Roediger & Karpicke (2006), "Test-Enhanced Learning: Taking Memory Tests Improves Long-Term Retention" (*Psychological Science*) — retention is tied to the *retrieval attempt* itself, not to who authored the question, so this reuses `06b`'s real evidence-linked questions as the in-loop test rather than synthesizing new ones with the QG model (`09`). Karpicke & Roediger (2008), "The Critical Importance of Retrieval for Learning" (*Science*) — repeated *successful* retrieval predicts retention better than repeated exposure, which is why a passing probe is left alone (no wasted retry) and a failing one gets another retrieval attempt, not just another pass over the same text.

**ML framing**: generate → verify → corrective-regenerate over narrow natural-language feedback (which answer span is missing), the same shape as Self-Refine (Madaan et al., 2023, arXiv:2303.17651) and Reflexion (Shinn et al., 2023, arXiv:2303.11366).

**Scope tonight**: curation only, mirroring `06b`'s own scope. Training a stage-2 checkpoint on these pairs (`07c`, mirroring `07b`) needs a GPU and is deferred.

**Cost gate default is unchanged from `06b`'s convention**: `CONFIRM_SPEND = False`, small `PILOT_LIMIT` — nothing here spends API budget until that is flipped by hand.

In [1]:
import json
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import pandas as pd
import yaml

from src.pipeline.curation import probe_accuracy_by_position, retention_probes
from src.pipeline.embeddings import embed_texts, load_config as load_embed_config
from src.pipeline.rate_limit import RateLimiter
from src.pipeline.teacher import curate_document_threaded_with_testing, load_curation_config

CFG = load_curation_config()
EMBED_CFG = load_embed_config()
CHUNK_MAX_WORDS = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))["max_words"]
OUT_DIR = Path("data/processed/rehearsal_testing_effect_stage2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"teacher          : {CFG['model']} (temperature {CFG['temperature']}, timeout {CFG['timeout']}s)")
print(f"embedding        : {EMBED_CFG['model']}")
print(f"chunk max_words  : {CHUNK_MAX_WORDS}")
print(f"document shape   : {CFG['doc_chunks']} chunks x {CFG['docs_per_corpus']} docs/corpus")
print(f"genres           : {CFG['genres']}")
print(f"output dir       : {OUT_DIR}")

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


teacher          : meta/llama-3.1-70b-instruct (temperature 0.0, timeout 120.0s)
embedding        : nvidia/llama-nemotron-embed-1b-v2
chunk max_words  : 200
document shape   : 12 chunks x 6 docs/corpus
genres           : ['news', 'narrativeqa', 'caselaw', 'wiki']
output dir       : data/processed/rehearsal_testing_effect_stage2


## 1. Corpora and the split

Identical to `06b` — same 20 corpora/genre, same disjoint-from-eval split, same corpus-level split discipline.

In [2]:
GENRES = CFG["genres"]
N_TRAIN, N_VAL, N_TEST = CFG["n_train_corpora"], CFG["n_val_corpora"], CFG["n_test_corpora"]

splits = {"train": [], "val": [], "test": []}
for genre in GENRES:
    dirs = sorted((root / "data" / "processed" / f"{genre}_train").glob("corpus_*"))
    if not dirs:
        raise SystemExit(f"no corpora for {genre} — run data/scripts/build_{genre}_train_corpora.py")
    splits["train"] += dirs[:N_TRAIN]
    splits["val"] += dirs[N_TRAIN:N_TRAIN + N_VAL]
    splits["test"] += dirs[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]

for name, dirs in splits.items():
    print(f"{name:5}: {len(dirs)} corpora")

train: 56 corpora
val  : 12 corpora
test : 12 corpora


## 2. Pseudo-documents (identical to `06b`)

Same `chunk_with_spans`/`build_documents` — each pseudo-document already carries `answers_by_chunk`, the exact input the testing-effect loop needs, so nothing new is required here.

In [3]:
def chunk_with_spans(text: str, max_words: int) -> list[dict]:
    """Consecutive `max_words`-word windows with their absolute char spans."""
    starts, cursor = [], 0
    for word in text.split():
        i = text.find(word, cursor)
        starts.append(i)
        cursor = i + len(word)
    return [
        {
            "text": text[starts[k]:(starts[k + max_words] if k + max_words < len(starts) else len(text))].strip(),
            "char_start": starts[k],
            "char_end": starts[k + max_words] if k + max_words < len(starts) else len(text),
        }
        for k in range(0, len(starts), max_words)
    ]


def build_documents(corpus_dir: Path) -> list[dict]:
    """`docs_per_corpus` documents of `doc_chunks` consecutive chunks, evenly
    spaced, each carrying the answer spans whose evidence lands in it."""
    text = (corpus_dir / "corpus.txt").read_text(encoding="utf-8")
    chunks = chunk_with_spans(text, CHUNK_MAX_WORDS)

    extractive_path = corpus_dir / "questions_extractive.csv"
    questions_path = extractive_path if extractive_path.exists() else corpus_dir / "questions.csv"
    questions = pd.read_csv(questions_path)
    questions = questions.dropna(subset=["answer", "evidence_char_pos"]) if "answer" in questions.columns else questions.iloc[0:0]

    doc_chunks, n_docs = CFG["doc_chunks"], CFG["docs_per_corpus"]
    if len(chunks) < doc_chunks:
        return []
    stride = max(doc_chunks, (len(chunks) - doc_chunks) // max(1, n_docs - 1)) if n_docs > 1 else doc_chunks
    starts = [s for s in range(0, len(chunks) - doc_chunks + 1, stride)][:n_docs]

    documents = []
    for doc_index, start in enumerate(starts):
        window = chunks[start:start + doc_chunks]
        answers_by_chunk = {}
        for position, chunk in enumerate(window):
            hits = questions[
                (questions["evidence_char_pos"] >= chunk["char_start"])
                & (questions["evidence_char_pos"] < chunk["char_end"])
            ] if len(questions) else questions
            answers_by_chunk[position] = [str(a) for a in hits["answer"].tolist()] if len(questions) else []
        documents.append({
            "key": f"{corpus_dir.parent.name}/{corpus_dir.name}/doc{doc_index:02d}",
            "genre": corpus_dir.parent.name.replace("_train", ""),
            "chunk_texts": [c["text"] for c in window],
            "answers_by_chunk": answers_by_chunk,
        })
    return documents


documents = {name: [d for c in dirs for d in build_documents(c)] for name, dirs in splits.items()}
for name, docs in documents.items():
    n_answers = sum(len(a) for d in docs for a in d["answers_by_chunk"].values())
    print(f"{name:5}: {len(docs):4} documents, {len(docs) * CFG['doc_chunks']:5} chunks, {n_answers:5} answer spans")

train:  336 documents,  4032 chunks,  9780 answer spans
val  :   72 documents,   864 chunks,  1971 answer spans
test :   72 documents,   864 chunks,  1916 answer spans


## 3. Cost gate

Same discipline as `06b` §3 — small `PILOT_LIMIT` first, check §5's probe pass/retry/fallback stats on it, then raise. Each in-loop test adds up to `max_retries` extra teacher calls **only for chunks that have linked answers *and* fail the first probe** — most chunks cost exactly what `06b`'s plain curation already costs.

In [4]:
CACHE_PATH = OUT_DIR / "curation_cache.jsonl"
CONFIRM_SPEND = True  # <- flip to True to actually call the teacher
PILOT_LIMIT = 20  # <- curate at most this many NEW documents this run; None = no cap
MAX_RETRIES = 2
PROBE_F1_THRESHOLD = CFG["probe_f1_threshold"]

cached = {}
if CACHE_PATH.exists():
    for line in CACHE_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            record = json.loads(line)
            cached[record["key"]] = record


def interleave_by_genre(docs: list[dict]) -> list[dict]:
    from itertools import zip_longest
    by_genre: dict[str, list[dict]] = {}
    for d in docs:
        by_genre.setdefault(d["genre"], []).append(d)
    return [d for group in zip_longest(*by_genre.values()) for d in group if d is not None]


from itertools import zip_longest

train_remaining = [d for d in interleave_by_genre(documents["train"]) if d["key"] not in cached]
val_interleaved = interleave_by_genre(documents["val"])
test_interleaved = interleave_by_genre(documents["test"])
val_test_remaining = [
    (split_name, d)
    for (val_d, test_d) in zip_longest(val_interleaved, test_interleaved)
    for split_name, d in (("val", val_d), ("test", test_d))
    if d is not None and d["key"] not in cached
]
todo_all = [("train", d) for d in train_remaining] + val_test_remaining
todo = todo_all[:PILOT_LIMIT] if PILOT_LIMIT is not None else todo_all

from collections import Counter

print(f"cached        : {len(cached)} documents")
print(f"remaining     : {len(todo_all)} documents")
print(f"this run      : {len(todo)} documents = {len(todo) * CFG['doc_chunks']} teacher calls (+ corrective retries) "
      f"+ {2 * len(todo) * CFG['doc_chunks']} embedding calls")
print(f"this run by split+genre: {dict(Counter((split_name, d['genre']) for split_name, d in todo))}")
print(f"\nCONFIRM_SPEND = {CONFIRM_SPEND}   PILOT_LIMIT = {PILOT_LIMIT}   MAX_RETRIES = {MAX_RETRIES}")

todo = [d for _, d in todo]

if todo and not CONFIRM_SPEND:
    print("-> set CONFIRM_SPEND = True and re-run this cell's successor to start")

cached        : 5 documents
remaining     : 475 documents
this run      : 20 documents = 240 teacher calls (+ corrective retries) + 480 embedding calls
this run by split+genre: {('train', 'news'): 6, ('train', 'caselaw'): 5, ('train', 'wiki'): 5, ('train', 'narrativeqa'): 4}

CONFIRM_SPEND = True   PILOT_LIMIT = 20   MAX_RETRIES = 2


## 4. Curate

Same parallel-documents / sequential-within-a-document structure as `06b` §4, `curate_document_threaded_with_testing` in place of `curate_document_threaded`. The extra fields (`probe_passed`, `retry_counts`, `fell_back_to_verbatim`) get cached alongside the usual ones.

In [5]:
if not CONFIRM_SPEND:
    print("skipped — CONFIRM_SPEND is False")
else:
    import threading
    from concurrent.futures import ThreadPoolExecutor, as_completed

    from tqdm.auto import tqdm

    MAX_WORKERS = 4
    rate_limiter = RateLimiter(CFG["requests_per_second"])

    progress_lock = threading.Lock()
    total_chunks = sum(len(document["chunk_texts"]) for document in todo)
    chunk_bar = tqdm(total=total_chunks, desc="chunks (all in-flight documents)")

    def _on_chunk_done(position, total):
        with progress_lock:
            chunk_bar.update(1)

    def _curate(document):
        result = curate_document_threaded_with_testing(
            document["chunk_texts"],
            {int(k): v for k, v in document["answers_by_chunk"].items()},
            EMBED_CFG, CFG, embed_fn=embed_texts,
            max_retries=MAX_RETRIES, probe_f1_threshold=PROBE_F1_THRESHOLD,
            on_chunk_done=_on_chunk_done, rate_limiter=rate_limiter,
        )
        return document, result

    with CACHE_PATH.open("a", encoding="utf-8") as cache_file:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = [pool.submit(_curate, document) for document in todo]
            for future in tqdm(as_completed(futures), total=len(futures), desc="documents"):
                document, result = future.result()
                record = {
                    "key": document["key"],
                    "genre": document["genre"],
                    "chunk_texts": document["chunk_texts"],
                    "answers_by_chunk": {str(k): v for k, v in document["answers_by_chunk"].items()},
                    "gists": result.gists,
                    "context_texts": result.context_texts,
                    "write_back_actions": result.write_back_actions,
                    "memory_states": result.memory_states,
                    "failed_positions": result.failed_positions,
                    "retrieval_failed_positions": result.retrieval_failed_positions,
                    "store_failed_positions": result.store_failed_positions,
                    "probe_passed": result.probe_passed,
                    "retry_counts": result.retry_counts,
                    "fell_back_to_verbatim": result.fell_back_to_verbatim,
                }
                cache_file.write(json.dumps(record, ensure_ascii=False) + "\n")
                cache_file.flush()
                cached[document["key"]] = record
    chunk_bar.close()

    failures = sum(len(r["failed_positions"]) for r in cached.values())
    embed_failures = sum(
        len(r["retrieval_failed_positions"]) + len(r["store_failed_positions"]) for r in cached.values()
    )
    print(f"\ncurated {len(cached)} documents, {failures} failed teacher calls, {embed_failures} failed embedding calls")

chunks (all in-flight documents): 100%|██████████| 240/240 [1:46:59<00:00, 26.75s/it]


curated 25 documents, 13 failed teacher calls, 0 failed embedding calls


## 5. Did testing-effect curation actually change anything?

Two separate questions, since a mechanism that fires but does not move the outcome is not evidence for the outcome:

1. **Does the in-loop mechanism actually do anything?** — probe pass rate on first try / after retry / fallback rate. If the fallback rate is ~0, either the teacher rarely loses facts in the first place at this document scale, or the probe threshold is too lenient — either way the mechanism has little room to matter and the retention comparison below should be read as inconclusive, not negative.
2. **Does it raise final retention over `06b`'s plain curation?** — same `retention_probes`/`probe_accuracy_by_position` read as `06b` §5, run over `memory_states` (post-testing) here vs. `06b`'s cache for the same documents where both exist.

In [6]:
records = list(cached.values())

if not records:
    print("no cached documents yet — run §3-4 with CONFIRM_SPEND = True first.")
else:
    n_chunks_with_answers = sum(1 for r in records for p in r["probe_passed"] if p is not None)
    n_passed_first_try = sum(
        1 for r in records
        for p, retries in zip(r["probe_passed"], r["retry_counts"])
        if p is not None and p and retries == 0
    )
    n_passed_after_retry = sum(
        1 for r in records
        for p, retries in zip(r["probe_passed"], r["retry_counts"])
        if p is not None and p and retries > 0
    )
    n_fell_back = sum(1 for r in records for f in r["fell_back_to_verbatim"] if f)
    total_retries = sum(sum(r["retry_counts"]) for r in records)

    print(f"chunks with linked answers to test : {n_chunks_with_answers}")
    print(f"  passed first try                 : {n_passed_first_try} ({n_passed_first_try / max(1, n_chunks_with_answers):.1%})")
    print(f"  passed after corrective retry     : {n_passed_after_retry} ({n_passed_after_retry / max(1, n_chunks_with_answers):.1%})")
    print(f"  fell back to verbatim             : {n_fell_back} ({n_fell_back / max(1, n_chunks_with_answers):.1%})")
    print(f"total corrective-retry calls spent  : {total_retries}")

    probes = []
    for record in records:
        answers_by_chunk = {int(k): v for k, v in record["answers_by_chunk"].items()}
        probes += retention_probes(record["memory_states"], answers_by_chunk, PROBE_F1_THRESHOLD)
    report = probe_accuracy_by_position(probes, collapse_after=3)
    print(f"\ntesting-effect curation retention: early {report['early_accuracy']:.3f}  "
          f"late {report['late_accuracy']:.3f}  drop {report['drop']:+.3f}")

    # Compare against 06b's plain curation on the *same* document keys, if
    # its cache is present — same documents, same probes, only the curation
    # mechanism differs, so this isolates the testing-effect loop's own
    # contribution rather than any difference in which documents got curated.
    plain_cache_path = Path("data/processed/rehearsal_elaborative_stage2/curation_cache.jsonl")
    if plain_cache_path.exists():
        plain_cached = {}
        for line in plain_cache_path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = json.loads(line)
                plain_cached[r["key"]] = r
        shared_keys = [r["key"] for r in records if r["key"] in plain_cached]
        if shared_keys:
            plain_probes = []
            for key in shared_keys:
                r = plain_cached[key]
                answers_by_chunk = {int(k): v for k, v in r["answers_by_chunk"].items()}
                plain_probes += retention_probes(r["memory_states"], answers_by_chunk, PROBE_F1_THRESHOLD)
            plain_report = probe_accuracy_by_position(plain_probes, collapse_after=3)
            print(f"06b plain curation, same {len(shared_keys)} documents: early {plain_report['early_accuracy']:.3f}  "
                  f"late {plain_report['late_accuracy']:.3f}  drop {plain_report['drop']:+.3f}")
        else:
            print("\nno overlapping document keys with 06b's cache yet -- pseudo-document sampling is "
                  "deterministic given the same corpora/config, so this should fill in once both caches "
                  "cover the same PILOT_LIMIT range.")
    else:
        print(f"\n(no {plain_cache_path} to compare against)")

chunks with linked answers to test : 142
  passed first try                 : 30 (21.1%)
  passed after corrective retry     : 92 (64.8%)
  fell back to verbatim             : 20 (14.1%)
total corrective-retry calls spent  : 133

testing-effect curation retention: early 0.863  late 0.676  drop +0.186
06b plain curation, same 25 documents: early 0.387  late 0.458  drop -0.071


## Summary

_To be filled in after a real (`CONFIRM_SPEND = True`) run._

What decides whether this is worth training a stage-2 checkpoint on (`07c`, needs a GPU — deferred, see intro):

1. **Fallback rate near 0%** → the mechanism rarely engages at this document/chunk scale; the comparison in §5 will be underpowered regardless of the retention numbers. Consider a smaller `max_slot_sentences` / smaller chunks / a stricter `PROBE_F1_THRESHOLD` to give it more to actually correct, or treat the near-0 rate itself as the finding (plain curation already retains most evidence-linked facts at this scale, so testing-effect has little room to help *here* — it might still matter at a longer accumulated-compression horizon `06b`'s own diagnostic already flags collapsing).
2. **Retention drop lower than `06b`'s plain-curation figure on the same documents** → the mechanism is doing what it is meant to; worth scaling to full curation and training `07c`.
3. **Retention drop roughly unchanged** → the corrective loop is fixing individual chunks' *local* recoverability without changing the *rolling* collapse shape stage 2 exists to fix — plausible, since `probe_passed`/retries operate at write-back time, not across the later revisions that come after a slot has already passed its own probe once. Worth checking whether facts that passed their own chunk's probe still survive later ages, same as `06b`'s original diagnostic already does.